# Credit Risk Prediction - Main Project HubThis is the central notebook for the credit risk prediction project. Use this to access all analyses, run scripts, and interact with the model.## Project Overview**Goal**: Predict loan default risk with interpretable counterfactual explanations**Key Results**:- Performance: 89.6% AUC-ROC, 0.081 Brier Score- Fairness: Passes disparate impact and equalized odds criteria- Calibration: 0.3% gap between predicted and actual rates- Interpretability: 96% of rejected cases have actionable counterfactuals---

## Quick Navigation### Analysis Notebooks (in order)1. [Data Cleaning](notebooks/data_cleaning.ipynb) - Preprocessing and feature engineering2. [Exploratory Analysis](notebooks/EDA.ipynb) - Data exploration and visualization3. [Feature Analysis](notebooks/feature_analysis.ipynb) - Feature importance4. [Model Training](notebooks/mlp_training.ipynb) - Deep learning model training5. [Model Evaluation](notebooks/model_evaluation.ipynb) - Performance comparison### Interactive Demos- [SHOWCASE Notebook](SHOWCASE.ipynb) - Interactive predictions and counterfactuals### Reports- [Technical Report](PROJECT_REPORT.md) - 5,300-word comprehensive report- [Project Structure](PROJECT_STRUCTURE.md) - File organization guide---

## Run All AnalysesExecute the cells below to run key analyses:

In [ ]:
import warningswarnings.filterwarnings('ignore')print("Libraries loaded")

### 1. Model Performance Summary

In [ ]:
import jsonimport pandas as pd# Load metricswith open('results/mlp_metrics.json', 'r') as f:    metrics = json.load(f)print("="*70)print("MODEL PERFORMANCE")print("="*70)print(f"\nTest Set Metrics:")print(f"  AUC-ROC: {metrics['test_metrics']['auc_roc']:.4f}")print(f"  AUC-PR: {metrics['test_metrics']['auc_pr']:.4f}")print(f"  Brier Score: {metrics['test_metrics']['brier_score']:.4f}")# Calibrationpreds = pd.read_csv('results/mlp_predictions.csv')actual = preds['true_label'].mean()predicted = preds['predicted_probability_calibrated'].mean()print(f"\nCalibration:")print(f"  Actual default rate: {actual:.1%}")print(f"  Predicted rate: {predicted:.1%}")print(f"  Gap: {abs(actual - predicted):.1%}")

### 2. Feature Importance AnalysisRun this to generate feature importance visualization:

In [ ]:
# Run feature importance analysisimport subprocessimport sysprint("Running feature importance analysis...")result = subprocess.run([sys.executable, 'feature_importance.py'],                       capture_output=True, text=True)print(result.stdout)if result.stderr:    print("Errors:", result.stderr)

### 3. Bias and Fairness AnalysisRun this to perform comprehensive bias analysis:

In [ ]:
# Run bias analysisprint("Running bias analysis...")result = subprocess.run([sys.executable, 'bias_analysis.py'],                       capture_output=True, text=True)print(result.stdout)if result.stderr:    print("Errors:", result.stderr)

### 4. Generate Counterfactual ExplanationsRun this to generate counterfactual explanations for high-risk cases:

In [ ]:
# Run DiCE counterfactual generationprint("Generating counterfactual explanations...")print("This may take several minutes...")result = subprocess.run([sys.executable, 'dice_setup.py'],                       capture_output=True, text=True, timeout=600)print(result.stdout)if result.stderr:    print("Errors:", result.stderr)

### 5. View Results Summary

In [ ]:
# Summarize all resultsprint("="*70)print("RESULTS SUMMARY")print("="*70)# Model performanceprint("\n1. MODEL PERFORMANCE")print(f"   AUC-ROC: {metrics['test_metrics']['auc_roc']:.4f}")print(f"   Calibration Gap: {abs(actual - predicted):.1%}")# Bias analysistry:    bias_gender = pd.read_csv('results/bias_gender.csv')    max_gap = bias_gender['Calibration Gap'].max()    print(f"\n2. FAIRNESS")    print(f"   Gender calibration: Max gap {max_gap:.1%}")    with open('results/bias_analysis.json', 'r') as f:        bias = json.load(f)    print(f"   All groups: Well-calibrated")except FileNotFoundError:    print("\n2. FAIRNESS")    print("   Run bias analysis first (Section 3)")# Feature importancetry:    features = pd.read_csv('results/top_features.csv')    print(f"\n3. TOP RISK FACTORS")    print(f"   {features.iloc[0]['Feature']}: {features.iloc[0]['Coefficient']:.2f}")    print(f"   {features.iloc[1]['Feature']}: {features.iloc[1]['Coefficient']:.2f}")    print(f"   {features.iloc[2]['Feature']}: {features.iloc[2]['Coefficient']:.2f}")except FileNotFoundError:    print("\n3. TOP RISK FACTORS")    print("   Run feature importance first (Section 2)")# Counterfactualstry:    cf_summary = pd.read_csv('results/dice_counterfactuals/verification_summary.csv')    print(f"\n4. COUNTERFACTUALS")    print(f"   Flip rate: {cf_summary['flip_rate'].mean():.1%}")    print(f"   High-risk cases analyzed: {len(cf_summary)}")except FileNotFoundError:    print("\n4. COUNTERFACTUALS")    print("   Run counterfactual generation first (Section 4)")print("\n" + "="*70)

---## Interactive Prediction DemoUse the SHOWCASE notebook for interactive predictions. Or run a quick demo below:

In [ ]:
# Quick prediction demoimport torchimport torch.nn as nnimport joblibimport numpy as np# Load modelclass ResidualBlock(nn.Module):    def __init__(self, dim, dropout=0.3):        super(ResidualBlock, self).__init__()        self.block = nn.Sequential(            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.ReLU(), nn.Dropout(dropout),            nn.Linear(dim, dim), nn.BatchNorm1d(dim)        )        self.relu = nn.ReLU()        self.dropout = nn.Dropout(dropout)    def forward(self, x):        return self.dropout(self.relu(self.block(x) + x))class SimpleMLP(nn.Module):    def __init__(self, input_dim):        super(SimpleMLP, self).__init__()        self.input_layer = nn.Sequential(nn.Linear(input_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.5))        self.res_block1 = ResidualBlock(512, dropout=0.4)        self.res_block2 = ResidualBlock(512, dropout=0.4)        self.down1 = nn.Sequential(nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4))        self.res_block3 = ResidualBlock(256, dropout=0.3)        self.down2 = nn.Sequential(nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3))        self.res_block4 = ResidualBlock(128, dropout=0.2)        self.down3 = nn.Sequential(nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2))        self.output_layer = nn.Sequential(nn.Linear(64, 1), nn.Sigmoid())    def forward(self, x):        x = self.input_layer(x)        x = self.res_block1(x); x = self.res_block2(x); x = self.down1(x)        x = self.res_block3(x); x = self.down2(x); x = self.res_block4(x)        x = self.down3(x); return self.output_layer(x)checkpoint = torch.load('models/mlp_model.pth', map_location='cpu', weights_only=False)model = SimpleMLP(checkpoint['input_dim'])model.load_state_dict(checkpoint['model_state_dict'])model.eval()calibrator = joblib.load('models/calibrator.pkl')print("Model loaded successfully")print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Example predictiontest = pd.read_csv('data/test.csv')sample = test.iloc[0:1].drop(columns=['status']).valueswith torch.no_grad():    pred_uncal = model(torch.FloatTensor(sample)).numpy()[0, 0]    pred_cal = calibrator.predict_proba(pred_uncal.reshape(-1, 1))[0, 1]print("\nExample Prediction:")print(f"Sample from test set")print(f"Calibrated Default Probability: {pred_cal:.1%}")print(f"Decision: {'REJECTED' if pred_cal >= 0.5 else 'APPROVED'}")print(f"\nFor interactive demos with custom inputs, use SHOWCASE.ipynb")

---## Project Files Overview### Data Files- `data/Loan_Default.csv` - Original dataset (148,670 samples)- `data/train.csv`, `val.csv`, `test.csv` - Split datasets### Models- `models/mlp_model.pth` - Trained neural network- `models/calibrator.pkl` - Platt scaling calibrator- `models/preprocessor.pkl` - Feature scaler### Results- `results/mlp_predictions.csv` - Test predictions- `results/mlp_metrics.json` - Performance metrics- `results/bias_*.csv` - Fairness analysis- `results/top_features.csv` - Feature importance- `results/dice_counterfactuals/` - Counterfactual explanations### Scripts- `feature_importance.py` - Feature importance analysis- `bias_analysis.py` - Fairness evaluation- `dice_setup.py` - Counterfactual generation### Documentation- `README.md` - Project overview- `PROJECT_REPORT.md` - Technical report (5,300 words)- `PROJECT_STRUCTURE.md` - File organization---## Next Steps1. **For Analysis**: Run notebooks 1-5 in sequence2. **For Demo**: Open SHOWCASE.ipynb for interactive predictions3. **For Results**: Run sections 2-4 above to generate all analyses4. **For Report**: Convert PROJECT_REPORT.md to Word using pandoc---## ContactFor questions or issues, refer to the project README.